# C14b — Grad-CAM : test de fidélité par suppression (E7)

La carte Grad-CAM du CRNN montre-t-elle vraiment ce qui fait décider le réseau ? Pour chaque
extrait de test :

1. carte Grad-CAM de la classe prédite (`echolens.core.models.grad_cam`, le code même de
   l'application : sortie du bloc 4, décision C14) ;
2. on masque de plus en plus de cases du spectrogramme standardisé (une case = un point de la
   carte, 16 bandes × 160 ms), soit **les plus chaudes d'abord**, soit **au hasard** ; une case
   masquée vaut 0, la moyenne du train après la standardisation R4 (décision C14) ;
3. on suit la probabilité de la classe prédite au départ. La carte est fidèle si elle tombe
   nettement plus vite avec le premier ordre : l'aire sous la courbe est plus petite.

Données jamais vues par le réseau testé :

- **ESC-50** : le CRNN de production a appris les 5 folds ; on prend donc, pour chaque fold k,
  le CRNN de C9 du tour k (graine 42, même recette), qui n'a jamais vu le fold k (P3) ;
- **émotions** : le CRNN de production (C12, graine 42) sur le test de CREMA-D et RAVDESS,
  acteurs jamais vus (P1), fenêtre centrale de 3 s comme dans C12.

Paramètres : `configs/grad_cam_fidelity.yaml`. Résultat : `c14_grad_cam_fidelity.json`, à
copier dans `training/results/` par une PR.

## a. Démarrage

In [ ]:
import sys

BRANCH = "main"
REPO_URL = "https://github.com/mentalist011101/TP_SIGNAL_DS.git"
CLONE_DIR = "/kaggle/working/TP_SIGNAL_DS"

!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}
# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.
%pip install -q --ignore-requires-python -e {CLONE_DIR}

sys.path.insert(0, CLONE_DIR + "/src")
print(sys.version)

## b. GPU et réglages communs

In [ ]:
import json
import subprocess
import time
from pathlib import Path

import numpy as np
import torch

from echolens.config import CONFIG_DIR
from echolens.core.data.class_labels import load_class_labels
from echolens.core.features import FeatureConfig
from echolens.training.evaluation import EvaluationConfig
from echolens.training.family_a import load_family_a_config
from echolens.training.grad_cam_fidelity import load_fidelity_config

if not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU : activer l'accélérateur GPU T4 du notebook.")
device = torch.device("cuda", 0)
GPU_NAME = torch.cuda.get_device_name(device)
print("GPU :", GPU_NAME, "· torch", torch.__version__)
NOTEBOOK_START = time.time()

commit = subprocess.run(
    ["git", "-C", CLONE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
fidelity_config = load_fidelity_config()
evaluation_config = EvaluationConfig.from_yaml(CONFIG_DIR / "evaluation.yaml")
family_a_config = load_family_a_config()
feature_config = FeatureConfig.from_yaml(CONFIG_DIR / "features.yaml")
labels = load_class_labels()
emotion_ids = [class_label.id for class_label in labels.emotions]
esc50_ids = [class_label.id for class_label in labels.esc50_classes]
OUTPUT_DIR = Path("/kaggle/working")
print(fidelity_config)

## c. Entrées

Sorties attachées au notebook : `echolens-c9-family-a-esc50-crnn` (poids des CRNN de chaque
tour, `c9/weights/crnn_fold<k>_seed42.pt`, avec leurs statistiques), `echolens-c9-family-a-esc50`
(log-Mel d'ESC-50) et la sortie de C12 (`emotion_log_mels.npz`). Le CRNN de production des
émotions est téléchargé depuis Hugging Face, à la révision figée de `configs/registry.yaml`.

In [ ]:
INPUT_DIR = Path("/kaggle/input")
FOLD_COUNT = 5
SEED = 42


def find_one(pattern: str) -> Path:
    # Un seul fichier doit correspondre, sinon on ne sait pas lequel on mesure.
    matches = sorted(INPUT_DIR.glob(pattern))
    if len(matches) != 1:
        raise RuntimeError(f"{len(matches)} fichiers pour {pattern} : il en faut exactement un")
    return matches[0]


ESC50_LOG_MELS_PATH = find_one("**/esc50_log_mels.npz")
EMOTION_LOG_MELS_PATH = find_one("**/emotion_log_mels.npz")
# Les poids de la session « -crnn » (courbe complète, PR #25), pas ceux du premier calcul.
C9_WEIGHTS = {}
for fold in range(1, FOLD_COUNT + 1):
    pattern = f"**/echolens-c9-family-a-esc50-crnn/c9/weights/crnn_fold{fold}_seed{SEED}.pt"
    C9_WEIGHTS[fold] = find_one(pattern)
print("log-Mel ESC-50 :", ESC50_LOG_MELS_PATH)
print("log-Mel émotions :", EMOTION_LOG_MELS_PATH)
for fold, path in C9_WEIGHTS.items():
    print("CRNN du tour", fold, ":", path)

## d. ESC-50 : le CRNN du tour k sur le fold k

Chaque extrait est standardisé avec les statistiques de l'entraînement de SON réseau (R4).
L'accuracy affichée sert de contrôle : elle doit être proche de celle de C9 pour la graine 42
(`training/results/esc50_family_a_crnn.json`), sinon les poids ou les log-Mel ne vont pas
ensemble.

In [ ]:
from echolens.training.family_a import fold_indices, load_log_mels, standardize
from echolens.training.grad_cam_fidelity_models import deletion_curves, load_c9_network

esc50_set = load_log_mels(ESC50_LOG_MELS_PATH)
esc50_parts = {"class_indices": [], "hottest_first": [], "random": [], "labels": [], "folds": []}
esc50_accuracy = {}
start = time.time()
for fold in range(1, FOLD_COUNT + 1):
    # Étape 1 : le réseau du tour et ses statistiques, sur le GPU.
    network, statistics = load_c9_network(C9_WEIGHTS[fold], family_a_config.network)
    network = network.to(device)
    # Étape 2 : les 400 extraits du fold de test de ce tour, standardisés comme à l'entraînement.
    indices = fold_indices(esc50_set.folds, [fold])
    standardized = standardize(esc50_set.log_mels[indices], statistics)
    # Étape 3 : les deux courbes de chaque extrait.
    curves = deletion_curves(network, standardized, fidelity_config)
    fold_labels = esc50_set.labels[indices]
    esc50_accuracy[fold] = float(np.mean(curves.class_indices == fold_labels))
    esc50_parts["class_indices"].append(curves.class_indices)
    esc50_parts["hottest_first"].append(curves.hottest_first)
    esc50_parts["random"].append(curves.random)
    esc50_parts["labels"].append(fold_labels)
    esc50_parts["folds"].append(np.full(len(indices), fold))
    print(f"fold {fold} : {len(indices)} extraits, accuracy {esc50_accuracy[fold]:.3f}")
esc50_seconds = time.time() - start
esc50_arrays = {name: np.concatenate(parts) for name, parts in esc50_parts.items()}
print(f"ESC-50 : {esc50_seconds / 60:.1f} min")

## e. Émotions : le CRNN de production sur le test (acteurs jamais vus)

Même entrée que dans C12 et dans l'application : fenêtre centrale de 3 s, complétée par du
silence (−100 dB, R2) si l'extrait est plus court, puis standardisation avec les statistiques
du config.json du modèle (R4). Contrôle : l'empreinte de `configs/features.yaml` doit être celle
de l'entraînement (R5), et l'accuracy proche de celle du registre.

In [ ]:
from huggingface_hub import hf_hub_download

from echolens.core.frame_windows import center_windows, window_frame_count
from echolens.core.models.production_config import (
    CONFIG_FILE,
    WEIGHTS_FILE,
    config_fingerprint,
    load_production_config,
)
from echolens.core.models.registry import ModelRegistry
from echolens.training.family_a_emotions import load_emotion_log_mels, split_indices
from echolens.training.grad_cam_fidelity_models import load_production_crnn

EMOTION_MODEL_ID = "echolens-emotion-crnn"
registry = ModelRegistry.from_yaml(CONFIG_DIR / "registry.yaml")
entry = registry.get(EMOTION_MODEL_ID)
repo_id = entry.weights.removeprefix("hf://")
config_path = Path(hf_hub_download(repo_id, CONFIG_FILE, revision=entry.revision))
weights_path = Path(hf_hub_download(repo_id, WEIGHTS_FILE, revision=entry.revision))
production_config = load_production_config(config_path)
if production_config.features_sha256 != config_fingerprint(CONFIG_DIR / "features.yaml"):
    raise RuntimeError("configs/features.yaml a changé depuis l'entraînement du CRNN (R5)")
if production_config.labels != emotion_ids:
    raise RuntimeError("Les classes du modèle ne sont pas celles de configs/labels.yaml")
network, statistics = load_production_crnn(config_path, weights_path)
network = network.to(device)

# Étape 1 : fenêtre centrale de chaque extrait de test.
emotion_set = load_emotion_log_mels(EMOTION_LOG_MELS_PATH)
test_indices = split_indices(emotion_set, "test")
window_frames = window_frame_count(production_config.window_s, feature_config)
windows = center_windows(
    emotion_set.log_mels[test_indices],
    emotion_set.lengths[test_indices],
    window_frames,
    feature_config.db_floor,
)
# Étape 2 : standardisation R4, puis les deux courbes de chaque extrait.
standardized = standardize(windows, statistics)
start = time.time()
curves = deletion_curves(network, standardized, fidelity_config)
emotion_seconds = time.time() - start
emotion_labels = emotion_set.targets[test_indices]
emotion_accuracy = float(np.mean(curves.class_indices == emotion_labels))
emotion_arrays = {
    "class_indices": curves.class_indices,
    "hottest_first": curves.hottest_first,
    "random": curves.random,
    "labels": emotion_labels,
}
print(f"{len(test_indices)} extraits de test, accuracy {emotion_accuracy:.3f}")
print("Accuracy du registre :", entry.metrics)
print(f"Émotions : {emotion_seconds / 60:.1f} min")

## f. Résultats

Pour chaque tâche : courbes moyennes, aire moyenne sous chaque courbe, écart (aire au hasard −
aire des plus chaudes d'abord) avec son intervalle à 95 % par bootstrap sur les extraits (P8),
et part des extraits où la carte fait mieux que le hasard. Le JSON est à copier dans
`training/results/c14_grad_cam_fidelity.json` ; les courbes de chaque extrait restent en sortie
Kaggle (`c14/curves_*.npz`) pour les figures.

In [ ]:
import datetime

from echolens.training.grad_cam_fidelity import summarize_deletion


def task_summary(arrays: dict, class_ids: list[str]) -> dict:
    summary = summarize_deletion(
        arrays["hottest_first"], arrays["random"], fidelity_config, evaluation_config
    )
    summary["accuracy"] = float(np.mean(arrays["class_indices"] == arrays["labels"]))
    summary["classes"] = class_ids
    return summary


report = {
    "experiment": "c14_grad_cam_fidelity",
    "date": datetime.datetime.now(datetime.UTC).isoformat(),
    "branch": BRANCH,
    "commit": commit,
    "gpu": GPU_NAME,
    "config": fidelity_config.model_dump(),
    "sound_event": task_summary(esc50_arrays, esc50_ids),
    "emotion": task_summary(emotion_arrays, emotion_ids),
}
report["sound_event"]["models"] = "CRNN de C9, graine 42, tour k évalué sur le fold k (P3)"
report["sound_event"]["accuracy_per_fold"] = esc50_accuracy
report["emotion"]["models"] = f"{EMOTION_MODEL_ID} (révision {entry.revision}), test (P1)"
report["seconds"] = {"sound_event": esc50_seconds, "emotion": emotion_seconds}
report["notebook_minutes"] = (time.time() - NOTEBOOK_START) / 60

ARRAYS_DIR = OUTPUT_DIR / "c14"
ARRAYS_DIR.mkdir(parents=True, exist_ok=True)
np.savez(ARRAYS_DIR / "curves_sound_event.npz", **esc50_arrays)
np.savez(ARRAYS_DIR / "curves_emotion.npz", **emotion_arrays)
report_path = OUTPUT_DIR / "c14_grad_cam_fidelity.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

for task in ["sound_event", "emotion"]:
    summary = report[task]
    low, high = summary["area_difference_ci95"]
    print(
        f"{task} : aire plus chaudes d'abord {summary['area_hottest_first']:.3f}, "
        f"au hasard {summary['area_random']:.3f}, écart {summary['area_difference']:.3f} "
        f"[{low:.3f} ; {high:.3f}], carte meilleure sur "
        f"{summary['share_of_clips_hottest_first_lower']:.0%} des extraits"
    )

## g. Courbes moyennes (contrôle visuel)

In [ ]:
import shutil

import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for axis, task, title in [
    (axes[0], "sound_event", "Événements (ESC-50)"),
    (axes[1], "emotion", "Émotions (test)"),
]:
    summary = report[task]
    fractions = summary["fractions"]
    axis.plot(fractions, summary["mean_curve_hottest_first"], label="Plus chaudes d'abord")
    axis.plot(fractions, summary["mean_curve_random"], label="Au hasard")
    axis.set_title(title)
    axis.set_xlabel("Part des cases masquées")
axes[0].set_ylabel("Probabilité de la classe prédite")
axes[0].legend()
figure.tight_layout()
figure.savefig(ARRAYS_DIR / "deletion_curves.png", dpi=150)
plt.show()
shutil.rmtree(CLONE_DIR, ignore_errors=True)